In [0]:
# COMMAND ----------
# 02 - ISOLATION FOREST ANOMALY

import numpy as np
from sklearn.ensemble import IsolationForest

df = spark.table("fraud.scored_xgb").toPandas()

lower_threshold = df["invoice_count"].quantile(0.10)
df["scenario"] = "standard"
df.loc[df["amount"] > df["avg_amount"] * 2, "scenario"] = "expense"
df.loc[df["invoice_count"] <= lower_threshold, "scenario"] = "low_volume"

df["anomaly_flag"] = 0
df["anomaly_score"] = 0.0

iso_features = ["amount","avg_amount","invoice_count","velocity_spike","benford_score"]

for scenario, contamination in {
    "standard": 0.01,
    "expense": 0.05,
    "new_vendor": 0.10
}.items():
    subset = df[df["scenario"] == scenario]
    if len(subset) > 5:
        iso_model = IsolationForest(
            n_estimators=200,
            contamination=contamination,
            random_state=42
        )
        X_subset = subset[iso_features]
        iso_model.fit(X_subset)
        df.loc[subset.index, "anomaly_score"] = iso_model.decision_function(X_subset)
        df.loc[subset.index, "anomaly_flag"] = iso_model.predict(X_subset)

df["anomaly_flag"] = df["anomaly_flag"].apply(lambda x: 1 if x == -1 else 0)

spark.createDataFrame(df).write.mode("overwrite").saveAsTable("fraud.scored_xgb_iso")
